<a href="https://colab.research.google.com/github/ancestor9/mathematics-for-machine-learning/blob/main/pytorch_recap/Task_03_pretrained_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 사전 훈련(Pretrained) 모델이란? 🧠

사전 훈련 모델은 ImageNet과 같은 대규모 데이터셋으로 이미 학습이 완료된 모델입니다. ImageNet은 1,000개의 클래스를 가진 1,000,000장 이상의 이미지로 구성된 거대한 데이터셋입니다.

사전 훈련 모델을 사용하는 이유:

시간과 자원 절약: 모델을 처음부터 학습시키려면 수많은 데이터와 고성능 GPU가 필요합니다. 사전 훈련 모델을 사용하면 이 과정을 건너뛸 수 있습니다.

높은 성능: 대규모 데이터로 학습된 모델은 이미지의 특징을 매우 효과적으로 이해하고 있습니다. 이 모델을 재활용하면 소규모 데이터셋으로도 높은 성능을 얻을 수 있습니다.

일반화 능력: 다양한 이미지 데이터를 이미 경험했기 때문에, 새로운 데이터에 대한 일반화 능력이 뛰어납니다.



## 2. PyTorch에서 사전 훈련 모델 사용하기

PyTorch의 torchvision.models 모듈은 ImageNet으로 사전 훈련된 다양한 모델을 제공합니다. 대표적인 모델로는 ResNet, VGG, AlexNet 등이 있습니다.



Step 1: 사전 훈련 모델 불러오기

torchvision.models를 사용하여 ResNet18 모델을 불러와 봅시다. pretrained=True 옵션을 설정하면 ImageNet으로 미리 학습된 가중치를 불러옵니다.



In [ ]:
import torch
import torchvision.models as models

# ImageNet으로 사전 훈련된 ResNet-18 모델 불러오기
resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print(resnet18)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 142MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Step 2: 모델 구조 파악하기

사전 훈련 모델은 크게 특징 추출기(Feature Extractor)와 분류기(Classifier)로 구성됩니다.

특징 추출기: 합성곱 층들로 이루어져 있으며, 이미지의 시각적 특징을 추출합니다.

분류기: 마지막에 위치한 선형(Linear) 층으로, 추출된 특징을 바탕으로 이미지를 1,000개의 클래스 중 하나로 분류합니다.

우리가 원하는 것은 우리의 데이터셋(예: 고양이/강아지)을 분류하는 것이므로, 기존 모델의 분류기를 우리 목적에 맞게 수정해야 합니다.





Step 3: 분류기 수정하기

모델의 마지막 분류기 층을 우리의 새로운 문제에 맞게 변경합니다. 예를 들어, 고양이와 강아지 두 가지 클래스를 분류하려면 출력 뉴런 수를 2로 변경해야 합니다.



In [ ]:
# ResNet-18 모델의 마지막 분류기(fc) 층 확인
print(f"원래 ResNet18의 마지막 층: {resnet18.fc}")

# 고양이와 강아지 2개의 클래스 분류를 위해 출력 뉴런 수를 2로 변경
# ResNet의 마지막 층은 'fc'입니다.
num_ftrs = resnet18.fc.in_features
resnet18.fc = torch.nn.Linear(num_ftrs, 2) # 출력 뉴런을 2개로 변경

print(f"수정된 ResNet18의 마지막 층: {resnet18.fc}")

원래 ResNet18의 마지막 층: Linear(in_features=512, out_features=1000, bias=True)
수정된 ResNet18의 마지막 층: Linear(in_features=512, out_features=2, bias=True)


## 3. 전이 학습(Transfer Learning)과 파인튜닝(Fine-tuning)

사전 훈련 모델을 사용하는 방법은 크게 두 가지가 있습니다.

특징 추출기(Feature Extractor) 고정: 모델의 특징 추출기 부분은 그대로 두고, 마지막 분류기만 학습시키는 방법입니다. 기존 학습된 특징을 그대로 활용하므로 학습이 매우 빠릅니다.

전체 모델 파인튜닝(Fine-tuning): 모델의 모든 층을 우리의 데이터셋에 맞게 미세 조정(fine-tune)하는 방법입니다. 특징 추출기의 가중치까지 업데이트하므로 더 높은 성능을 기대할 수 있지만, 더 많은 데이터와 시간이 필요합니다.

일반적으로는 먼저 특징 추출기를 고정하고 학습시킨 후, 추가로 전체 모델을 미세 조정하는 방식을 사용합니다.



In [ ]:
# 모든 파라미터의 기울기 계산을 비활성화 (학습되지 않도록)
for param in resnet18.parameters():
    param.requires_grad = False

# 마지막 분류기 층의 파라미터만 학습 가능하도록 설정
for param in resnet18.fc.parameters():
    param.requires_grad = True

# 이제 optimizer에 resnet18.fc.parameters()만 전달하면 됩니다.
optimizer = torch.optim.Adam(resnet18.fc.parameters(), lr=0.001)

이렇게 하면 학습 과정에서 resnet18.fc의 가중치만 업데이트됩니다.

## 실습 문제
1. torchvision.models에서 ResNet18 외에 다른 사전 훈련 모델(예: VGG16, AlexNet)을 불러와서 모델의 마지막 층 구조를 확인해 보세요.

2. 사전 훈련 모델을 사용하는 것이 모델을 처음부터 학습시키는 것보다 왜 효율적인지 두 가지 이유를 들어 설명하세요.

3. 사전 훈련 모델의 features 부분과 classifier 부분의 역할을 구분하여 설명하세요.